In [1]:
import numpy as np
import math
from scipy.stats import t
from scipy.stats import erlang, expon

rng = np.random.default_rng(30)

Part 1

In [ ]:
def poisson_blocking_system(m, s, lbd, max, rng):
    '''
    m is the number of servers
    s is the mean number of services per unit of time
    lbd is the mean time between arrivals
    max is the total number of customer arrivals we want to simulate
    '''

    t = 0                   # clock of the simulation
    t_serv = np.zeros(m)    # remaining service time at eacher server
    arrivals = 0            # count the number of arrivals
    blocked = 0             # count the number of blocked customers

    while(arrivals < max):                              # loop until all the customers arrived

        arrivals += 1                                   # add a new arrival
        t_arr = rng.exponential(lbd)                    # time between the previous arrival and this one
        t += t_arr                                      # update the clock
        t_serv -= t_arr                                 # update the remaining service time for each server
        t_serv = np.clip(t_serv, a_min=0, a_max=None)   # ensure that there is no negative remaining service time

        mask = t_serv == 0                              # Is there a free server?
        if (np.sum(mask) == 0):
            blocked += 1                                # if no, then block the new customer
        else:
            for i in range(m):
                if (t_serv[i] == 0):
                    t_serv[i] = rng.exponential(s)      # if yes, it takes the customer and its time is updated
                    break
        
    frac_blocked = blocked/max                          # compute the fraction of blocked customers

    return frac_blocked

In [3]:
rng = np.random.default_rng(30)

n = 10
observations = [poisson_blocking_system(10, 8, 1, 10000, rng) for _ in range(n)]        # 10 observations
mean_frac_blocked = np.mean(observations)
print('Mean number of blocked customers =', mean_frac_blocked)

tdist = t.interval(0.95, df=n-1)
S = np.sqrt(1/(n-1) * (np.sum([observations[i]**2 for i in range(n)]) - n*mean_frac_blocked**2))
conf1 = tdist[0] * S / np.sqrt(n) + mean_frac_blocked
conf2 = tdist[1] * S / np.sqrt(n) + mean_frac_blocked
print(f'Confidence interval: {(conf1, conf2)}')

Mean number of blocked customers = 0.12254000000000001
Confidence interval: (np.float64(0.11841895365280357), np.float64(0.12666104634719644))


In [4]:
def analytical_blocking_system(m, s, lbd):
    A = s*lbd
    nom = (A**m)/(math.factorial(m))            # nominator of the formula on the last slide
    denom = 0
    for i in range(m+1):
        denom += (A**i)/(math.factorial(i))     # denominator of the formula
    B = nom/denom
    return B

analytical_blocking_system(10, 8, 1)

0.12166106425295149

Part 2

In [5]:
def erlang_blocking_system(m, s, k, max, rng):

    t = 0
    t_serv = np.zeros(m)
    arrivals = 0
    blocked = 0

    while(arrivals < max):

        arrivals += 1
        t_arr = erlang.rvs(a=k, scale=1/k)              # erlang distribution
        t += t_arr
        t_serv -= t_arr
        t_serv = np.clip(t_serv, a_min=0, a_max=None)

        mask = t_serv == 0
        if (np.sum(mask) == 0):
            blocked += 1
        else:
            for i in range(m):
                if (t_serv[i] == 0):
                    t_serv[i] = rng.exponential(s)
                    break
        
    frac_blocked = blocked/max

    return frac_blocked

In [6]:
rng = np.random.default_rng(30)

n = 10
observations = [erlang_blocking_system(10, 8, 2, 10000, rng) for _ in range(n)]
mean_frac_blocked = np.mean(observations)
print('Mean number of blocked customers =', mean_frac_blocked)

tdist = t.interval(0.95, df=n-1)
S = np.sqrt(1/(n-1) * (np.sum([observations[i]**2 for i in range(n)]) - n*mean_frac_blocked**2))
conf1 = tdist[0] * S / np.sqrt(n) + mean_frac_blocked
conf2 = tdist[1] * S / np.sqrt(n) + mean_frac_blocked
print(f'Confidence interval: {(conf1, conf2)}')

Mean number of blocked customers = 0.09154
Confidence interval: (np.float64(0.08741840179593284), np.float64(0.09566159820406715))


In [7]:
def hyperexponential_blocking_system(m, s, lbd, p, max, rng):

    t = 0
    t_serv = np.zeros(m)
    arrivals = 0
    blocked = 0

    while(arrivals < max):

        arrivals += 1
        choice = rng.choice(a=[0,1], p=p)
        t_arr = rng.exponential(1/lbd[choice])          # hyperexponential inter-arrival times
        t += t_arr
        t_serv -= t_arr
        t_serv = np.clip(t_serv, a_min=0, a_max=None)

        mask = t_serv == 0
        if (np.sum(mask) == 0):
            blocked += 1
        else:
            for i in range(m):
                if (t_serv[i] == 0):
                    t_serv[i] = rng.exponential(s)
                    break
        
    frac_blocked = blocked/max

    return frac_blocked

In [8]:
rng = np.random.default_rng(30)

n = 10
observations = [hyperexponential_blocking_system(10, 8, [0.8333, 5], [0.8, 0.2], 10000, rng) for _ in range(n)]
mean_frac_blocked = np.mean(observations)
print('Mean number of blocked customers =', mean_frac_blocked)

tdist = t.interval(0.95, df=n-1)
S = np.sqrt(1/(n-1) * (np.sum([observations[i]**2 for i in range(n)]) - n*mean_frac_blocked**2))
conf1 = tdist[0] * S / np.sqrt(n) + mean_frac_blocked
conf2 = tdist[1] * S / np.sqrt(n) + mean_frac_blocked
print(f'Confidence interval: {(conf1, conf2)}')

Mean number of blocked customers = 0.13871
Confidence interval: (np.float64(0.1355576551564441), np.float64(0.1418623448435559))


Part 3

In [9]:
def constant_poisson_blocking_system(m, s, lbd, max, rng):

    t = 0
    t_serv = np.zeros(m)
    arrivals = 0
    blocked = 0

    while(arrivals < max):

        arrivals += 1
        t_arr = rng.exponential(lbd)
        t += t_arr
        t_serv -= t_arr
        t_serv = np.clip(t_serv, a_min=0, a_max=None)

        mask = t_serv == 0
        if (np.sum(mask) == 0):
            blocked += 1
        else:
            for i in range(m):
                if (t_serv[i] == 0):
                    t_serv[i] = s               # constant
                    break
        
    frac_blocked = blocked/max

    return frac_blocked

In [10]:
rng = np.random.default_rng(30)

n = 10
observations = [constant_poisson_blocking_system(10, 8, 1, 10000, rng) for _ in range(n)]        # 10 observations
mean_frac_blocked = np.mean(observations)
print('Mean number of blocked customers =', mean_frac_blocked)

tdist = t.interval(0.95, df=n-1)
S = np.sqrt(1/(n-1) * (np.sum([observations[i]**2 for i in range(n)]) - n*mean_frac_blocked**2))
conf1 = tdist[0] * S / np.sqrt(n) + mean_frac_blocked
conf2 = tdist[1] * S / np.sqrt(n) + mean_frac_blocked
print(f'Confidence interval: {(conf1, conf2)}')

Mean number of blocked customers = 0.12196
Confidence interval: (np.float64(0.11833627809516568), np.float64(0.1255837219048343))


In [11]:
def pareto_poisson_blocking_system(m, k, lbd, max, rng):

    t = 0
    t_serv = np.zeros(m)
    arrivals = 0
    blocked = 0

    while(arrivals < max):

        arrivals += 1
        t_arr = rng.exponential(lbd)
        t += t_arr
        t_serv -= t_arr
        t_serv = np.clip(t_serv, a_min=0, a_max=None)

        mask = t_serv == 0
        if (np.sum(mask) == 0):
            blocked += 1
        else:
            for i in range(m):
                if (t_serv[i] == 0):
                    t_serv[i] = rng.pareto(k)               # pareto
                    break
        
    frac_blocked = blocked/max

    return frac_blocked

In [13]:
rng = np.random.default_rng(30)

n = 10
print('For k=', 1.05)
observations = [pareto_poisson_blocking_system(10, 1.05, 1, 10000, rng) for _ in range(n)]        # 10 observations
mean_frac_blocked = np.mean(observations)
print('Mean number of blocked customers =', mean_frac_blocked)

tdist = t.interval(0.95, df=n-1)
S = np.sqrt(1/(n-1) * (np.sum([observations[i]**2 for i in range(n)]) - n*mean_frac_blocked**2))
conf1 = tdist[0] * S / np.sqrt(n) + mean_frac_blocked
conf2 = tdist[1] * S / np.sqrt(n) + mean_frac_blocked
print(f'Confidence interval: {(conf1, conf2)}')

For k= 1.05
Mean number of blocked customers = 0.07646
Confidence interval: (np.float64(0.045028075354123204), np.float64(0.1078919246458768))


In [14]:
rng = np.random.default_rng(30)

n = 10
print('For k=', 2.05)
observations = [pareto_poisson_blocking_system(10, 2.05, 1, 10000, rng) for _ in range(n)]        # 10 observations
mean_frac_blocked = np.mean(observations)
print('Mean number of blocked customers =', mean_frac_blocked)

tdist = t.interval(0.95, df=n-1)
S = np.sqrt(1/(n-1) * (np.sum([observations[i]**2 for i in range(n)]) - n*mean_frac_blocked**2))
conf1 = tdist[0] * S / np.sqrt(n) + mean_frac_blocked
conf2 = tdist[1] * S / np.sqrt(n) + mean_frac_blocked
print(f'Confidence interval: {(conf1, conf2)}')

For k= 2.05
Mean number of blocked customers = 0.0
Confidence interval: (np.float64(0.0), np.float64(0.0))


In [17]:
def uniform_poisson_blocking_system(m, down, up, lbd, max, rng):

    t = 0
    t_serv = np.zeros(m)
    arrivals = 0
    blocked = 0

    while(arrivals < max):

        arrivals += 1
        t_arr = rng.exponential(lbd)
        t += t_arr
        t_serv -= t_arr
        t_serv = np.clip(t_serv, a_min=0, a_max=None)

        mask = t_serv == 0
        if (np.sum(mask) == 0):
            blocked += 1
        else:
            for i in range(m):
                if (t_serv[i] == 0):
                    t_serv[i] = rng.uniform(down,up)               # uniform
                    break
        
    frac_blocked = blocked/max

    return frac_blocked

In [18]:
rng = np.random.default_rng(30)

n = 10
observations = [uniform_poisson_blocking_system(10, 0, 15, 1, 10000, rng) for _ in range(n)]        # 10 observations
mean_frac_blocked = np.mean(observations)
print('Mean number of blocked customers =', mean_frac_blocked)

tdist = t.interval(0.95, df=n-1)
S = np.sqrt(1/(n-1) * (np.sum([observations[i]**2 for i in range(n)]) - n*mean_frac_blocked**2))
conf1 = tdist[0] * S / np.sqrt(n) + mean_frac_blocked
conf2 = tdist[1] * S / np.sqrt(n) + mean_frac_blocked
print(f'Confidence interval: {(conf1, conf2)}')

Mean number of blocked customers = 0.09949
Confidence interval: (np.float64(0.09661124761169068), np.float64(0.10236875238830931))


In [ ]:
def normal_poisson_blocking_system(m, mean, std, lbd, max, rng):

    t = 0
    t_serv = np.zeros(m)
    arrivals = 0
    blocked = 0

    while(arrivals < max):

        arrivals += 1
        t_arr = rng.exponential(lbd)
        t += t_arr
        t_serv -= t_arr
        t_serv = np.clip(t_serv, a_min=0, a_max=None)

        mask = t_serv == 0
        if (np.sum(mask) == 0):
            blocked += 1
        else:
            for i in range(m):
                if (t_serv[i] == 0):
                    t_serv[i] = rng.normal(mean, std)            # normal
                    break
        
    frac_blocked = blocked/max

    return frac_blocked

In [ ]:
rng = np.random.default_rng(30)

n = 10
observations = [normal_poisson_blocking_system(10, 8, 2, 1, 10000, rng) for _ in range(n)]        # 10 observations
mean_frac_blocked = np.mean(observations)
print('Mean number of blocked customers =', mean_frac_blocked)

tdist = t.interval(0.95, df=n-1)
S = np.sqrt(1/(n-1) * (np.sum([observations[i]**2 for i in range(n)]) - n*mean_frac_blocked**2))
conf1 = tdist[0] * S / np.sqrt(n) + mean_frac_blocked
conf2 = tdist[1] * S / np.sqrt(n) + mean_frac_blocked
print(f'Confidence interval: {(conf1, conf2)}')

Part 4